In [6]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

In [7]:
conn = sqlite3.connect('data/checking-logs.sqlite')

In [8]:
project1 = pd.io.sql.read_sql(
    '''
    SELECT uid, numTrials, timestamp FROM checker 
    WHERE uid LIKE 'user_%' AND status = 'ready' AND labname = 'project1'
    ORDER BY uid;
    ''',
    conn, parse_dates=['timestamp']
)

project1['date'] = project1['timestamp'].dt.date
daily = project1.groupby(['uid', 'date'])['numTrials'].max().reset_index()
sorted_dates = sorted(daily['date'].unique())
daily['date_num'] = daily['date'].map({d: i + 1 for i, d in enumerate(sorted_dates)})

daily = daily.sort_values(['uid', 'date_num'])
daily

,uid,date,numTrials,date_num
0,user_1,2020-05-14,11,18
1,user_10,2020-05-12,7,16
2,user_10,2020-05-13,21,17
3,user_10,2020-05-14,59,18
4,user_11,2020-05-03,1,7
...,...,...,...,...
89,user_4,2020-05-13,137,17
90,user_4,2020-05-14,164,18
91,user_6,2020-05-13,1,17
92,user_6,2020-05-14,2,18


In [9]:
users = sorted(daily['uid'].unique())
all_dates = list(range(1, 20))

user_series = {}
for user in users:
    ud = daily[daily['uid'] == user].set_index('date_num')['numTrials']
    full = ud.reindex(range(0, 20), fill_value=0).astype(float)
    for d in range(1, 20):
        if full[d] == 0:
            full[d] = full[d - 1]
    user_series[user] = full

frames = []
for d in all_dates:
    frame_data = []
    for user in users:
        s = user_series[user]
        xs = list(range(0, d + 1))
        ys = [s[i] for i in xs]
        frame_data.append(go.Scatter(
            x=xs, y=ys, mode='lines+markers', name=user,
            legendgroup=user
        ))
    frames.append(go.Frame(data=frame_data, name=str(d)))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title='Dynamic of commits per user in project1',
        height=550,
        xaxis=dict(range=[1, 20], dtick=2, tick0=2),
        yaxis=dict(range=[0, 170], dtick=20),
        legend=dict(font=dict(size=10), tracegroupgap=1),
        updatemenus=[dict(type='buttons', showactive=False,
                          buttons=[dict(label='play', method='animate',
                                        args=[None, dict(frame=dict(duration=200,
                                                                    redraw=True),
                                                                    fromcurrent=True)]),
            ]
        )]
    )
)

fig.show()

In [10]:
conn.close()